In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F


device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [1]:
with open('../data/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print("total examples:", len(text))
print("vocab size:", vocab_size)
print(text[:25])

total examples: 1115394
vocab size: 65
First Citizen:
Before we 


In [4]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

block_size = 4
batch_size = 64
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(low=0, high=len(data)-block_size, size=(batch_size, ))

    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

Xb, Yb = get_batch('train')
print(Xb[0])
print(Yb[0])

tensor([46, 39, 58,  1], device='cuda:0')
tensor([39, 58,  1, 42], device='cuda:0')


In [5]:
n_embd = 16
tok_emb = nn.Embedding(vocab_size, n_embd)
pos_emb = nn.Embedding(block_size, n_embd)
unembed = nn.Linear(n_embd, vocab_size)
tok_emb.to(device)
pos_emb.to(device)
unembed.to(device)
Xb, Yb = get_batch('train')

x = tok_emb(Xb) + pos_emb(torch.arange(block_size, device=device))
logits = unembed(x)
B,T,C = logits.shape
logits = logits.view(B*T, C)
targets = Yb.view(B*T)
loss = F.cross_entropy(logits, targets)
loss   


tensor(4.4030, device='cuda:0', grad_fn=<NllLossBackward0>)

In [6]:
n_embd = 16
class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.unembed = nn.Linear(n_embd, vocab_size)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        
        tok_emb = self.tok_emb(idx) # (B,T,C) -> (batch, block, embed)
        pos_emb = self.pos_emb(torch.arange(T, device=device)) # (block, embed)
        x = tok_emb + pos_emb
        logits = self.unembed(x)
        
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)    
    
        return logits, loss


    def generate(self, idx, max_tokens=1):
        for _ in range(max_tokens):
            logits, _ = self(idx[:,-block_size:])
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=1)
            ix = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, ix), dim=1) # lol no need to cut
        return idx

In [ ]:
max_iters = 500
lr = 3e-3

model = GPT()
model.to(device)

model.load_state_dict(state_dict=torch.load("v1.pt", weights_only=True))
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
for iter in range(max_iters):  
    Xb, Yb = get_batch('train')
    logits, loss = model(Xb, Yb)
    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if iter % (max_iters // 5) == 0 or iter == max_iters - 1:
        print(loss.item())
torch.save(model.state_dict(), "v1.pt")

2.3886656761169434
2.36011004447937
2.46224308013916
2.4110522270202637
2.520897150039673
2.508445978164673


In [111]:
#model.generate()
# context = torch.ones((1, block_size), dtype=int, device=device)
# context = torch.tensor([encode(text[:block_size])], device=device)
context = torch.ones((1,1), dtype=torch.long, device=device)
for _ in range(10):
    logits, loss = model(context[:,-block_size:])
    logits = logits[:, -1, :]
    probs = F.softmax(logits, dim=1)
    ix = torch.multinomial(probs, num_samples=1)
    context = torch.cat((context, ix), dim=1) # lol no need to cut
decode(context[0].tolist())

' aith thern'

In [24]:
max_tokens = 2
idx = torch.ones((1,1), dtype=torch.long, device=device)
for _ in range(max_tokens):
    logits, _ = model(idx[:,-block_size:])
    logits = logits[:, -1, :]
    probs = F.softmax(logits, dim=1)
    ix = torch.multinomial(probs, num_samples=1)
    idx = torch.cat((idx, ix), dim=1) # lol no need to cut
print(decode(idx[0].tolist()))
print(idx.shape)

 wo
torch.Size([1, 3])


In [ ]:
context = torch.ones((1,1), dtype=torch.long, device=device)
output = model.generate(context, max_tokens=100)
print(decode(output[0].tolist()))

" t cogongan'tst ERWe m gu he tous\nStsse ganen ond m usicechiss tuthe the at my gio mbu, f VONu.\n\nEERI"